In [2]:
import numpy as np
import pandas as pd
from astropy.io import fits
import warnings

import glob, os, sys, timeit
import matplotlib
import numpy as np

#from pyqsofit.PyQSOFit import QSOFit
from astropy.io import fits
from astropy.table import Table
import matplotlib.pyplot as plt
import warnings

from astropy.coordinates import SkyCoord

#from speclite import filters

from astropy import units as u
from astropy.table import Table


warnings.filterwarnings("ignore")

#QSOFit.set_mpl_style()

from astroquery.sdss import SDSS
from astropy.table import Table


import os
from astroquery.sdss import SDSS
from astropy.coordinates import SkyCoord
import astropy.units as u


In [4]:
sample_df = pd.read_csv("data/aug1_chisq2_ebv_sn_allfields.csv")

with fits.open('data/dr16q_prop_May01_2024.fits') as hdul:
    data_cat = hdul[1].data
    #data_cat = data_cat[data_cat['Z_SYS'] <1]

    coords_sdss = SkyCoord(
        ra=data_cat['RA'],
        dec=data_cat['DEC'],
        unit=(u.deg, u.deg),
        frame='icrs'
    )

    # ...existing code...
    # Ensure all object_ids are strings and stripped
    sample_df['object_id'] = sample_df['object_id'].astype(str).str.strip()
    #obj_mean_corr_map = {str(obj['object_id']).strip(): obj['mean_corrected'] for obj in objs}

    coords_agn = SkyCoord(
        ra=sample_df['ra'],
        dec=sample_df['dec'],
        unit=(u.deg, u.deg),
        frame='icrs'
    )

    # Match using search_around_sky to find all pairs within a certain separation
    max_sep = 2.0 * u.arcsec  # maximum separation for matching

    idx_sdss, idx_agn, sep2d, _ = coords_agn.search_around_sky(coords_sdss, max_sep)

    # Store matches in a DataFrame for further analysis
    matches_df = pd.DataFrame({
        'agn_idx': idx_agn,
        'sdss_idx': idx_sdss,
        'sep_arcsec': sep2d.arcsec
    })

    # Only keep data_cat rows where there is a match
    matched_sdss_indices = np.unique(idx_sdss)
    data_cat = data_cat[matched_sdss_indices]
    # Add apparent_mag_2500 from agn_df into data_cat by matching indices
    # Assume agn_df has a column 'apparent_mag_2500' and matches to data_cat via matched_sdss_indices

    # Only keep rows in agn_df that correspond to matched_sdss_indices
    sample_df_matched = sample_df.iloc[idx_agn].reset_index(drop=True)

    # Add apparent_mag_2500 from agn_df_matched to data_cat as a new column
    # Assumes agn_df_matched['apparent_mag_2500'] exists and matches the order of data_cat after filtering


    # Convert data_cat to Table if not already
    if not isinstance(data_cat, Table):
        data_cat = Table(data_cat)



In [11]:
def fetch_spectrum_fits(sdss_name, plate, fiber, mjd, cache_dir="data/spectra_cache"):
    """
    Fetch SDSS spectrum for a given index in data_cat.
    Downloads and saves to FITS file if not already cached.
    """
    

    os.makedirs(cache_dir, exist_ok=True)

    # sdss_name = data_cat['SDSS_NAME'][i]
    # plate, fiber, mjd = data_cat['PLATE'][i], data_cat['FIBERID'][i], data_cat['MJD'][i]
    cache_file = os.path.join(cache_dir, f"{sdss_name}_p{plate}_f{fiber}_m{mjd}.fits")

    # Return cached FITS if available
    if os.path.exists(cache_file):
        from astropy.io import fits
        # print(f"Loaded cached spectrum for {sdss_name} from {cache_file}")
        return fits.open(cache_file, memmap=False), True

    # Download spectrum    
    spec = SDSS.get_spectra(plate=plate, fiberID=fiber, mjd=mjd)
    if spec is None or len(spec) == 0:
        raise ValueError(f"No spectrum found for index SDSS_NAME {sdss_name}")

    data = spec[0]  # First HDUList
    
    # Save to FITS file
    data.writeto(cache_file, overwrite=True)
    #print(f"Spectrum saved to {cache_file}")
    
    return data

In [ ]:
from tqdm import tqdm

SDSS.clear_cache()

N = len(data_cat)
spectrum_errors = []
for i, d in enumerate(tqdm(data_cat[:N], desc="Fetching spectra")):
    plate, fiber, mjd = d['PLATE'], d['FIBERID'], d['MJD']
    sdss_name = d['SDSS_NAME']
    # print(f"Fetching spectrum for index {i}/{N} with SDSS_NAME: {sdss_name}")
    # print("Plate:", plate, "FiberID:", fiber, "MJD:", mjd)
    try:
        fetch_spectrum_fits(sdss_name, plate, fiber, mjd)
    except Exception as e:
        # print(f"Error fetching spectrum for index {i}, SDSS_NAME {sdss_name}: {e}")
        spectrum_errors.append((i, sdss_name, str(e)))
        continue
print(f"Fetched {i+1} spectra successfully.")
print(len(spectrum_errors))

Fetching spectra:  52%|█████▏    | 6693/12908 [2:30:23<1:53:43,  1.10s/it] 

In [15]:
len(spectrum_errors)

9

In [ ]:
def fetch_spectrum_fits(sdss_name, plate, fiber, mjd, cache_dir="data/spectra_cache"):
    """
    Fetch SDSS spectrum for a given index in data_cat.
    Downloads and saves to FITS file if not already cached.
    """
    os.makedirs(cache_dir, exist_ok=True)

    # sdss_name = data_cat['SDSS_NAME'][i]
    # plate, fiber, mjd = data_cat['PLATE'][i], data_cat['FIBERID'][i], data_cat['MJD'][i]
    cache_file = os.path.join(cache_dir, f"{sdss_name}_p{plate}_f{fiber}_m{mjd}.fits")

    # Return cached FITS if available
    if os.path.exists(cache_file):
        from astropy.io import fits
        # print(f"Loaded cached spectrum for {sdss_name} from {cache_file}")
        return fits.open(cache_file, memmap=False), True

    # Download spectrum    
    spec = SDSS.get_spectra(plate=plate, fiberID=fiber, mjd=mjd)
    if spec is None or len(spec) == 0:
        raise ValueError(f"No spectrum found for index SDSS_NAME {sdss_name}")

    data = spec[0]  # First HDUList
    
    # Save to FITS file
    data.writeto(cache_file, overwrite=True)
    #print(f"Spectrum saved to {cache_file}")
    
    return data

In [10]:
from astroquery.sdss import SDSS
SDSS.clear_cache()

In [9]:
d = data_cat[8]
sdss_name = d['SDSS_NAME']
plate, fiber, mjd = d['PLATE'], d['FIBERID'], d['MJD']
spec = SDSS.get_spectra(plate=plate, fiberID=fiber, mjd=mjd)
spec

[[<astropy.io.fits.hdu.image.PrimaryHDU object at 0x7fbe45eb7b30>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45ed10a0>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45ee7350>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45efa690>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45f16390>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45d360c0>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45d49f10>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45d61d60>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45d75b20>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45d88a70>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45da07d0>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45db8560>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45dcc380>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45de4140>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45de7f20>, <astropy.

In [ ]:
plate, fiber, mjd

(9403, 485, 58018)

In [8]:
spec

[<astropy.io.fits.hdu.image.PrimaryHDU object at 0x7fbe468d3830>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe48915460>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe46901a30>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe4672d580>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe467e1190>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe467e3fe0>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe467fbdd0>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe468dfe60>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45e2a840>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45e3a5a0>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45e4e390>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45e620f0>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45e75ee0>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45e89d30>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fbe45ea1ac0>, <astropy.i

In [19]:
i = spectrum_errors[0]

(8,
 '000011.96+000225.2',
 'No spectrum found for index SDSS_NAME 000011.96+000225.2')